# Plot Distance vs. Correlation

Does a plot's correlation with MODIS weaken the further it sits from
the tower/pixel center? This is the same distance-decay idea validated
externally in `HF365_ControlPoints_and_Geolocation.ipynb` (far-away
control points correlate more weakly) -- applied here *within* the
existing 42-plot network itself, using each plot's actual distance from
the tower.

## IMPORTANT: this notebook currently runs on PLACEHOLDER coordinates
Real, precise plot coordinates were never obtained for this project --
early attempts (digitizing the HF069 site map by eye, then starting a
WebPlotDigitizer pass) never produced a finished, precise result. The
coordinates loaded below are a **reconstruction** based only on HF069's
documented transect structure (8 directions, 100m steps out to 500m) --
**not real survey data.** Treat every number in this notebook as
illustrative of the *method*, not yet a real finding. See this folder's
`status.md` for what's needed to make this real.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# vvv EDIT THIS once you have REAL plot coordinates vvv
PLOT_COORDS_CSV = "placeholder_plot_coords.csv"
# ^^^ EDIT THIS once you have REAL plot coordinates vvv

HF365_DIR = "."
HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}


## 1. Load plot coordinates and HF365 data

In [ ]:
coords = pd.read_csv(PLOT_COORDS_CSV).set_index("plot")
print(f"Loaded coordinates for {len(coords)} plots")
print(f"Distance range: {coords['dist_from_tower_m'].min():.0f}m to {coords['dist_from_tower_m'].max():.0f}m")

hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")


## 2. Each plot's correlation with the site average

Rather than correlating each plot against a single MODIS pixel (which
would require knowing exactly which pixel each plot falls in -- itself
dependent on precise coordinates), this uses each plot's correlation
with the **site-wide average** as the comparison target. A plot that's
further from the tower and behaves more differently from the rest of
the network should show up as more weakly correlated with that
average.

In [ ]:
def plot_vs_siteavg_correlation(df):
    site_avg = df.mean(axis=1)
    return df.apply(lambda col: col.corr(site_avg))

distance_corr_rows = []
for index_name, df in hf365_wide.items():
    corrs = plot_vs_siteavg_correlation(df)
    for plot, r in corrs.items():
        if plot in coords.index:
            distance_corr_rows.append(dict(
                index=index_name, plot=plot, correlation=r,
                distance_m=coords.loc[plot, "dist_from_tower_m"]
            ))

distance_corr_df = pd.DataFrame(distance_corr_rows)
print(f"Built {len(distance_corr_df)} (plot, index) distance/correlation pairs")
distance_corr_df.head()


## 3. Correlation vs. distance -- scatter + trend

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, index_name in zip(axes, HF365_FILES):
    sub = distance_corr_df[distance_corr_df["index"] == index_name]
    ax.scatter(sub["distance_m"], sub["correlation"], s=30, alpha=0.7)
    if len(sub) >= 2:
        coefs = np.polyfit(sub["distance_m"], sub["correlation"], 1)
        xs = np.linspace(sub["distance_m"].min(), sub["distance_m"].max(), 50)
        ax.plot(xs, np.polyval(coefs, xs), color="black", linewidth=1)
        r = sub["distance_m"].corr(sub["correlation"])
        ax.set_title(f"{index_name} (distance-vs-corr r={r:.2f})")
    ax.set_xlabel("Distance from tower (m)")
    ax.set_ylabel("Correlation with site average")
plt.tight_layout()
plt.show()


## 4. Binned view -- average correlation by distance band

Easier to read than the raw scatter: group plots into distance bands
and average their correlation within each band.

In [ ]:
bins = [0, 150, 250, 350, 450, 600]
labels = ["0-150m", "150-250m", "250-350m", "350-450m", "450m+"]
distance_corr_df["distance_band"] = pd.cut(distance_corr_df["distance_m"], bins=bins, labels=labels)

binned = distance_corr_df.groupby(["index", "distance_band"], observed=True)["correlation"].agg(["mean", "count"])
print(binned.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for index_name in HF365_FILES:
    sub = binned.loc[index_name]
    ax.plot(sub.index.astype(str), sub["mean"], marker="o", label=index_name)
ax.set_xlabel("Distance band from tower")
ax.set_ylabel("Mean correlation with site average")
ax.set_title("Correlation vs. distance band (placeholder coordinates)")
ax.legend()
plt.tight_layout()
plt.show()


## Summary

- This notebook demonstrates the **method** for testing whether
  plot-level correlation decays with distance from the tower --
  matching pixel/site data.
- **Every number above is currently based on placeholder/reconstructed
  coordinates, not real survey data** -- see `status.md` for what's
  needed to make this a real finding.
- Once real coordinates exist, re-running this notebook (just pointing
  `PLOT_COORDS_CSV` at the real file) is all that's needed -- no other
  code changes required.
